In [1]:
import pandas as pd
from sentence_transformers import CrossEncoder
from docx import Document

In [ ]:
top_candidates = pd.read_csv(
    "../outputs/faiss_top500.csv"
)

top_candidates.head()

,candidate_id,candidate_text,similarity_score
0,CAND_0058412,Current title: Senior Software Engineer (ML)\n...,0.745611
1,CAND_0051486,Current title: AI Specialist\nSummary: Data sc...,0.744759
2,CAND_0058152,Current title: Data Scientist\nSummary: Data s...,0.744204
3,CAND_0089012,Current title: AI Specialist\nSummary: Data sc...,0.742055
4,CAND_0063585,Current title: AI Research Engineer\nSummary: ...,0.740394


In [3]:
doc = Document(
    "../data/job_description.docx"
)

job_text = "\n".join(
    para.text
    for para in doc.paragraphs
)

In [4]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [5]:
pairs = [
    (job_text, text)
    for text in top_candidates["candidate_text"]
]

In [6]:
pairs[0]

('Job Description: Senior AI Engineer — Founding Team\nCompany: Redrob AI (Series A AI-native talent intelligence platform)\nLocation: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities\nEmployment Type: Full-time\nExperience Required: 5–9 years (see "what we mean by this" below)\n\nLet\'s be honest about this role\nWe\'re going to write this JD differently from most. We\'re a Series A company that just raised our round and we\'re building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we\'re going to tell you what we actually need and what we\'ve gotten wrong before.\nIf you\'ve spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn\'t it.\nIf you\'ve spent your career bouncing between early-stage startups and you want to "just code" with

In [7]:
scores = reranker.predict(
    pairs,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [8]:
top_candidates = top_candidates.copy()

top_candidates["cross_score"] = scores

In [9]:
top_candidates = top_candidates.sort_values(
    "cross_score",
    ascending=False
)

In [10]:
top_candidates[
    ["candidate_id",
     "similarity_score",
     "cross_score"]
].head(20)

,candidate_id,similarity_score,cross_score
59,CAND_0098952,0.719120,-2.717750
394,CAND_0052400,0.688049,-3.030254
2,CAND_0058152,0.744204,-3.061904
137,CAND_0076412,0.710414,-3.223301
95,CAND_0067403,0.714222,-3.377036
256,CAND_0033987,0.699800,-3.430194
224,CAND_0076995,0.702436,-3.445279
160,CAND_0021827,0.708003,-3.496102
299,CAND_0049978,0.696152,-3.569349
8,CAND_0084681,0.734335,-3.619651


In [11]:
top_candidates.to_csv(
    "../outputs/reranked_candidates.csv",
    index=False
)

In [12]:
top_100 = top_candidates.head(100)

top_100.head()

,candidate_id,candidate_text,similarity_score,cross_score
59,CAND_0098952,Current title: AI Research Engineer\nSummary: ...,0.719120,-2.717750
394,CAND_0052400,Current title: Computer Vision Engineer\nSumma...,0.688049,-3.030254
2,CAND_0058152,Current title: Data Scientist\nSummary: Data s...,0.744204,-3.061904
137,CAND_0076412,Current title: AI Research Engineer\nSummary: ...,0.710414,-3.223301
95,CAND_0067403,Current title: Junior ML Engineer\nSummary: Da...,0.714222,-3.377036


In [13]:
top_100.to_csv(
    "../outputs/top100_candidates.csv",
    index=False
)